# Setup

Creates database, schema, stage, compute pools, and optionally generates synthetic POC data.

**Prerequisites:** Update `poc/config.yaml` with your database, schema, and warehouse settings.

In [9]:
# 1. Setup & Imports
from snowflake.snowpark import Session
import numpy as np
import pandas as pd
from poc.utils import (
    get_connection_config, get_poc_data_config, 
    get_feature_config, get_training_config, 
    get_inference_config, generate_timestamps
)

conn_cfg = get_connection_config()
poc_cfg = get_poc_data_config()
feat_cfg = get_feature_config()
train_cfg = get_training_config()
infer_cfg = get_inference_config()

DB_NAME = conn_cfg["database"]
SCHEMA_DATA = conn_cfg["schema"]
STAGE_ARTIFACTS = conn_cfg["stage_artifacts"]
WAREHOUSE = conn_cfg["warehouse"]
NUM_FEATURES = poc_cfg["num_features"]

session = Session.builder.getOrCreate()
print(f"Connected to account: {session.get_current_account()}")
print(f"Current role: {session.get_current_role()}")

Connected to account: "SFSENORTHAMERICA-AFERAS_AWS1"
Current role: "ML_DEV_ROLE"


In [10]:
# 2. Configuration (loaded from config.yaml)
print(f"DB_NAME: {DB_NAME}")
print(f"SCHEMA_DATA: {SCHEMA_DATA}")
print(f"STAGE_ARTIFACTS: {STAGE_ARTIFACTS}")
print(f"NUM_FEATURES: {NUM_FEATURES}")


DB_NAME: MMT_POC
SCHEMA_DATA: FORECASTING
STAGE_ARTIFACTS: ML_STAGE
NUM_FEATURES: 5


In [11]:
# 4. Create Database and Schemas
session.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}").collect()
session.sql(f"USE DATABASE {DB_NAME}").collect()

session.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DATA}").collect()
session.sql(f"USE SCHEMA {SCHEMA_DATA}").collect()

print(f"✓ Database {DB_NAME} ready")
print(f"✓ Schema {SCHEMA_DATA} ready")

✓ Database MMT_POC ready
✓ Schema FORECASTING ready


In [12]:
# 4. Create Stages
session.sql(f"""CREATE STAGE IF NOT EXISTS {STAGE_ARTIFACTS}
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')""").collect()

print(f"✓ Stage @{STAGE_ARTIFACTS} ready")

✓ Stage @ML_STAGE ready


In [13]:

if poc_cfg["generate"]:

    feature_cols = ",\n    ".join([f"FEATURE_{i} FLOAT" for i in range(1, NUM_FEATURES + 1)])

    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {feat_cfg["name"]} (
        {feat_cfg["time_col"]} TIMESTAMP_NTZ,
        {feat_cfg["partition_col"]} VARCHAR(10),
        {feature_cols},
        {feat_cfg["target_col"]} FLOAT
    )
    """).collect()


    session.sql(f"""
        CREATE TABLE IF NOT EXISTS PREDICTIONS (
            {feat_cfg["partition_col"]} VARCHAR(10),
            {feat_cfg["time_col"]} TIMESTAMP_NTZ,
            {feat_cfg["target_col"]} FLOAT,
            PREDICTION FLOAT
        )
    """).collect()

    print(f"✓ Tables ready (FEATURE_TABLE with {NUM_FEATURES} features)")

else:
    print("Using existing features from table", feat_cfg["name"])

✓ Tables ready (FEATURE_TABLE with 5 features)


In [14]:
# 9. Configure warehouse
session.sql(f"""
    CREATE WAREHOUSE IF NOT EXISTS {WAREHOUSE}
    WITH
        WAREHOUSE_SIZE = MEDIUM; 
""").collect()

session.use_warehouse(WAREHOUSE)

for cfg in [train_cfg, infer_cfg]:
    if cfg["compute_pool_name"] in [r.name for r in session.sql("SHOW COMPUTE POOLS;").collect()]:
        print(f"Compute pool {cfg['compute_pool_name']} already exists:")
        session.sql(f"DESCRIBE COMPUTE POOL {cfg["compute_pool_name"]};").show()
    else:
        session.sql(f"""
            CREATE COMPUTE POOL {cfg["compute_pool_name"]}
            WITH
                MIN_NODES = 1
                MAX_NODES = {cfg["target_cluster_size"]}
                INSTANCE_FAMILY = {cfg["instance_family"]}
                AUTO_SUSPEND_SECS = 10;
        """).collect()

print(f"✓ Warehouse and compute pool ready.")

Compute pool MMT_POC_TRAIN_CPU_X64_S_5 already exists:
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"name"                     |"state"    |"min_nodes"  |"max_nodes"  |"instance_family"  |"num_services"  |"num_jobs"  |"auto_suspend_secs"  |"auto_resume"  |"active_nodes"  |"idle_nodes"  |"target_nodes"  |"created_on"                      |"resumed_on"                      |"updated_on"                      |"owner"      |"comment"  |"is_exclusive"  |"application"  |"error_code"  |"status_message"  |"placement_group"  |
-----------------------------------------------------------------------------------------------

In [15]:
# 10. Verify Setup
print("\n" + "="*60)
print("INFRASTRUCTURE SETUP COMPLETE")
print("="*60)

tables = session.sql("SHOW TABLES IN SCHEMA FORECASTING").collect()
print(f"\nTables created: {len(tables)}")
for t in tables:
    print(f"  • {t['name']}")

stages = session.sql("SHOW STAGES IN SCHEMA FORECASTING").collect()
print(f"\nStages created: {len(stages)}")
for s in stages:
    print(f"  • @{s['name']}")




INFRASTRUCTURE SETUP COMPLETE

Tables created: 2
  • FEATURE_TABLE
  • PREDICTIONS

Stages created: 1
  • @ML_STAGE


In [16]:
if poc_cfg["generate"]:

    NUM_PARTITIONS = poc_cfg["partitions"]
    TIMESTEP = poc_cfg["timestep"]
    NUM_TIMESTEPS = poc_cfg["num_timesteps"]
    NUM_FEATURES = poc_cfg["num_features"]
    START_DATE = poc_cfg["start_date"]

    print(f"POC Data Configuration:")
    print(f"   Partitions:    {NUM_PARTITIONS}")
    print(f"   Timestep:      {TIMESTEP}")
    print(f"   Num timesteps: {NUM_TIMESTEPS}")
    print(f"   Num features:  {NUM_FEATURES}")
    print(f"   Rows:          ~{NUM_PARTITIONS * NUM_TIMESTEPS:,}")

    print("Generating feature data with realistic patterns...")
    np.random.seed(42)

    records = []
    timestamps = generate_timestamps(START_DATE, TIMESTEP, NUM_TIMESTEPS)

    for partition_idx in range(NUM_PARTITIONS):
        partition_id = f"P{str(partition_idx).zfill(4)}"
        
        base_target = np.random.uniform(20, 100)
        partition_effect = np.random.uniform(0.7, 1.3)
        
        for ts in timestamps:
            doy = ts.timetuple().tm_yday
            dow = ts.weekday()
            
            seasonal = 1.0 + 0.2 * np.sin(2 * np.pi * doy / 365)
            weekday_effect = 1.1 if dow in (4, 5) else 1.0
            
            features = {}
            for i in range(1, NUM_FEATURES + 1):
                if i == 1:
                    features[f"FEATURE_{i}"] = round(np.random.normal(50, 10) * partition_effect, 4)
                elif i == 2:
                    features[f"FEATURE_{i}"] = round(np.random.normal(30, 5) * seasonal, 4)
                elif i == 3:
                    features[f"FEATURE_{i}"] = round(np.random.uniform(0, 100), 4)
                elif i == 4:
                    features[f"FEATURE_{i}"] = round(doy / 365.0, 4)
                elif i == 5:
                    features[f"FEATURE_{i}"] = round(dow / 6.0, 4)
                else:
                    features[f"FEATURE_{i}"] = round(np.random.normal(0, 1), 4)
            
            noise = np.random.normal(0, 3)
            target = base_target * partition_effect * seasonal * weekday_effect + noise
            for idx, (k, v) in enumerate(features.items()):
                weight = 0.3 / (idx + 1)
                target += weight * v
            target = max(0, round(target, 2))
            
            record = {
                feat_cfg["time_col"]: ts,
                feat_cfg["partition_col"]: partition_id,
                **features,
                feat_cfg["target_col"]: target,
            }
            records.append(record)

    print(f"Generated {len(records):,} rows")

    print(f"Uploading to {feat_cfg['name']}...")

    time_col = feat_cfg["time_col"]
    for i in range(0, len(records), 100_000):
        chunk = pd.DataFrame(records[i : i + 100_000])
        chunk[time_col] = pd.to_datetime(chunk[time_col])
        session.write_pandas(chunk, feat_cfg["name"], overwrite=(i == 0), auto_create_table=True, use_logical_type=True)
        print(f"  Uploaded {min(i + 100_000, len(records)):,} / {len(records):,}")

    print(f"{feat_cfg['name']} ready with {len(records):,} rows")

    print("\n" + "="*60)
    print("POC DATA GENERATION COMPLETE")
    print("="*60)

    part_col = feat_cfg["partition_col"]
    counts = session.sql(f"""
        SELECT 
            COUNT(*) AS TOTAL_ROWS,
            COUNT(DISTINCT {part_col}) AS PARTITIONS,
            MIN({time_col}) AS MIN_DATE,
            MAX({time_col}) AS MAX_DATE
        FROM {feat_cfg["name"]}
    """).collect()[0]

    print(f"\nData Summary:")
    print(f"  Total rows:  {counts['TOTAL_ROWS']:,}")
    print(f"  Partitions:  {counts['PARTITIONS']:,}")
    print(f"  Timestep:    {TIMESTEP}")
    print(f"  Num features: {NUM_FEATURES}")
    print(f"  Date range:  {counts['MIN_DATE']} to {counts['MAX_DATE']}")

POC Data Configuration:
   Partitions:    100
   Timestep:      day
   Num timesteps: 730
   Num features:  5
   Rows:          ~73,000
Generating feature data with realistic patterns...
Generated 73,000 rows
Uploading to FEATURE_TABLE...
  Uploaded 73,000 / 73,000
FEATURE_TABLE ready with 73,000 rows

POC DATA GENERATION COMPLETE

Data Summary:
  Total rows:  73,000
  Partitions:  100
  Timestep:    day
  Num features: 5
  Date range:  2024-03-02 00:00:00 to 2026-03-01 00:00:00
